In [0]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")

BRONZE = "nyc_taxi_trip.nyc.nyc_taxi"
SILVER = "nyc_taxi_trip.nyc.nyc_final"

In [0]:
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = [F.radians(c) for c in (lat1, lon1, lat2, lon2)]
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = (F.pow(F.sin(dlat / 2), 2)
         + F.cos(lat1) * F.cos(lat2) * F.pow(F.sin(dlon / 2), 2))
    return 6371 * 2 * F.asin(F.sqrt(a))

def manhattan_km(lat1, lon1, lat2, lon2):
    return (haversine_km(lat1, lon1, lat2, lon1) +
            haversine_km(lat1, lon1, lat1, lon2))

def bearing_deg(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = [F.radians(c) for c in (lat1, lon1, lat2, lon2)]
    dlon = lon2 - lon1
    y = F.sin(dlon) * F.cos(lat2)
    x = F.cos(lat1) * F.sin(lat2) - F.sin(lat1) * F.cos(lat2) * F.cos(dlon)
    return (F.degrees(F.atan2(y, x)) + 360) % 360

In [0]:
df = spark.table(BRONZE)

for c in ["fare_amount", "pickup_longitude", "pickup_latitude",
          "dropoff_longitude", "dropoff_latitude"]:
    df = df.withColumn(c, F.col(c).cast("double"))

# NYC bounding box
df = df.filter(
    F.col("pickup_longitude").between(-74.3, -73.6) &
    F.col("pickup_latitude").between(40.4, 41.0)
)

# drop NAs
critical_cols = ["pickup_datetime", "pickup_latitude", "pickup_longitude",
                 "dropoff_latitude", "dropoff_longitude",
                 "passenger_count", "fare_amount"]
df = df.dropna(subset=critical_cols)

df = df.withColumn("passenger_count", F.col("passenger_count").cast("int"))

# negative / zero fares
df = df.filter(F.col("fare_amount") > 0)

In [0]:
df = (df
    .withColumn("month", F.month("pickup_datetime"))
    .withColumn("haversine_km", haversine_km("pickup_latitude", "pickup_longitude",
                                             "dropoff_latitude", "dropoff_longitude"))
    .withColumn("manhattan_km", manhattan_km("pickup_latitude", "pickup_longitude",
                                             "dropoff_latitude", "dropoff_longitude"))
    .withColumn("bearing_deg", bearing_deg("pickup_latitude", "pickup_longitude",
                                           "dropoff_latitude", "dropoff_longitude"))
)

# outlier removal
df = df.filter(
    F.col("fare_amount").between(2.5, 200) &
    F.col("haversine_km").between(0, 100) &
    F.col("passenger_count").between(1, 6)
)

# time features
df = (df
    .withColumn("hour", F.hour("pickup_datetime"))
    .withColumn("hour_sin", F.sin(2 * F.lit(3.141592653589793) * F.col("hour") / 24))
    .withColumn("hour_cos", F.cos(2 * F.lit(3.141592653589793) * F.col("hour") / 24))
    # Spark: Sun=1..Sat=7  ->  pandas style: Mon=0..Sun=6
    .withColumn("day_of_week", (F.dayofweek("pickup_datetime") + 5) % 7)
    .withColumn("day_sin", F.sin(2 * F.lit(3.141592653589793) * F.col("day_of_week") / 7))
    .withColumn("day_cos", F.cos(2 * F.lit(3.141592653589793) * F.col("day_of_week") / 7))
    .withColumn("is_weekend", (F.col("day_of_week") >= 5).cast("int"))
)

In [0]:
(df.write
   .format("delta")
   .mode("overwrite")
   .saveAsTable(SILVER))

print(f"{spark.table(SILVER).count():,} rows in silver")

54,026,339 rows in silver


In [0]:
%sql
SELECT hour, ROUND(AVG(fare_amount), 2) AS avg_fare, COUNT(*) AS trips
FROM nyc_taxi_trip.nyc.nyc_final
GROUP BY hour ORDER BY hour

hour,avg_fare,trips
0,11.66,2139063
1,11.39,1584243
2,11.37,1181385
3,11.82,865563
4,13.56,627072
5,15.14,528226
6,12.16,1108495
7,10.98,1945639
8,10.9,2454114
9,10.85,2525542


In [0]:
%sql
SELECT is_weekend, ROUND(AVG(fare_amount), 2) AS avg_fare,
       ROUND(AVG(haversine_km), 2) AS avg_km
FROM nyc_taxi_trip.nyc.nyc_final
GROUP BY is_weekend

is_weekend,avg_fare,avg_km
1,11.27,3.43
0,11.34,3.27


In [0]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import xgboost as xgb
import mlflow

SILVER = "nyc_taxi_trip.nyc.nyc_final"

train_columns = [
    "haversine_km", "hour_sin", "hour_cos", "manhattan_km", "is_weekend",
    "day_sin", "day_cos", "passenger_count", "bearing_deg", "month"
]

pdf = (spark.table(SILVER)
       .select(*train_columns, "fare_amount")
       .sample(fraction=0.10, seed=123)
       .toPandas())

X = pdf[train_columns].to_numpy(dtype=np.float32)
y = pdf["fare_amount"].to_numpy(dtype=np.float32)
del pdf

X_train, X_val, y_train, y_val = train_test_split(X, y, random_state=123, test_size=0.2)
print(X_train.shape, X_val.shape)

(4320314, 10) (1080079, 10)


In [0]:
params = dict(
    n_estimators=600,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    device="cpu",
    objective="reg:squarederror",
    random_state=123,
    n_jobs=4,
)

with mlflow.start_run(run_name="xgb_taxi_sample"):
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train,
              eval_set=[(X_train, y_train), (X_val, y_val)],
              verbose=50)

    y_hat  = model.predict(X_train)
    y_pred = model.predict(X_val)

    metrics = {
        "mae_train": mean_absolute_error(y_train, y_hat),
        "mae_val":   mean_absolute_error(y_val, y_pred),
        "r2_train":  r2_score(y_train, y_hat),
        "r2_val":    r2_score(y_val, y_pred),
    }

    mlflow.log_params(params)
    mlflow.log_metrics(metrics)
    mlflow.log_param("sample_fraction", 0.10)
    mlflow.xgboost.log_model(model, "model", input_example=X_val[:5])

    print(metrics)

[0]	validation_0-rmse:9.20331	validation_1-rmse:9.21705
[50]	validation_0-rmse:4.32087	validation_1-rmse:4.36547
[100]	validation_0-rmse:4.20284	validation_1-rmse:4.25263
[150]	validation_0-rmse:4.17673	validation_1-rmse:4.23339
[200]	validation_0-rmse:4.16324	validation_1-rmse:4.22575
[250]	validation_0-rmse:4.15248	validation_1-rmse:4.22178
[300]	validation_0-rmse:4.14273	validation_1-rmse:4.21800
[350]	validation_0-rmse:4.13493	validation_1-rmse:4.21548
[400]	validation_0-rmse:4.12707	validation_1-rmse:4.21421
[450]	validation_0-rmse:4.11989	validation_1-rmse:4.21372
[500]	validation_0-rmse:4.11351	validation_1-rmse:4.21372
[550]	validation_0-rmse:4.10772	validation_1-rmse:4.21375
[599]	validation_0-rmse:4.10179	validation_1-rmse:4.21339


2026/11/01 20:21:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-31d50368-17d5.cloud.databricks.com/ml/experiments/1065245526760359/models/m-f99db36a3af94a5d804353493a1ad7a6?o=7474651565798627


{'mae_train': 2.0338845252990723, 'mae_val': 2.047001838684082, 'r2_train': 0.8165360689163208, 'r2_val': 0.8069071769714355}
